In [1]:
"""
train.py — Signature Verification  (ArcFace + FocalContrastive, v8.6)
======================================================================
BUGS FIXED vs v8.2
──────────────────
FIX #2  unwrap() was stripping torch.compile — model trained unoptimised.
        Now model_final is compiled HERE after device placement.

FIX #3  HardMiningLoss returned sum of 2 raw scalars with no normalisation.
        With lambda_hard=0.25 and batch=256 it dominated the loss entirely.
        Fixed by dividing by batch size so scale matches other losses.

FIX #4  EmbeddingMixupLoss used np.random.beta which is seeded separately
        from torch — broke full reproducibility.  Switched to torch.distributions.Beta.

ROOT CAUSE OF Loss=1.0000 ALWAYS (already fixed in v8.2):
  arc_norm = arc / arc.detach()  →  always 1.0
  con_norm = con / con.detach()  →  always 1.0
  total = 0.5×1.0 + 0.5×1.0    →  always 1.0
  Fix: use raw losses with lambdas that account for the 67× scale difference.
    lambda_arc=0.02 → 0.02×33 ≈ 0.66
    lambda_con=1.00 → 1.00×0.5 ≈ 0.50
"""

import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm

# model_final is the bare SiameseCNN — we compile it below after .to(device)
from model import model_final
from dataset import (
    train_pairs,  train_labels,
    val_pairs,    val_labels,
    test_pairs,   test_labels,
    image_cache,
    genuine_by_author,
    train_authors,          # needed for ArcFace author2id (no val/test leakage)
    train_transform,
    test_transform,
)


# ─────────────────────────────────────────────────────────────────────────────
# DEVICE + SEEDS
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name()}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True

N_AUG         = 15
CACHE_REFRESH = 5


# ─────────────────────────────────────────────────────────────────────────────
# FIX #2: compile HERE after device placement — not in model.py at import time.
# Safe fallback: torch.compile requires Triton (Linux only). On Windows or
# environments without Triton the compile call raises TritonMissing at the
# first forward pass, so we catch it and silently run the bare model instead.
# ─────────────────────────────────────────────────────────────────────────────
_base_model = model_final.to(device)

def _try_compile(model):
    """
    Compile with the best available backend:
      1. reduce-overhead  (inductor + Triton) — fastest, Linux/CUDA only
      2. aot_eager        (AOT graph capture, no Triton) — still faster than bare
      3. bare model       (last resort fallback)

    IMPORTANT: torch.compile() is LAZY — it returns a wrapper immediately and
    only triggers real compilation (and Triton) on the FIRST FORWARD PASS.
    A try/except around torch.compile() catches nothing; TritonMissing always
    fires inside train_epoch on batch 0 — which is why the previous fallback
    didn't work.

    Fix: explicitly import triton before choosing the backend so we never give
    an inductor-compiled model to the training loop when Triton is absent.
    """
    def _triton_available():
        try:
            import triton  # noqa: F401
            return True
        except ImportError:
            return False

    if _triton_available():
        try:
            compiled = torch.compile(model, mode="reduce-overhead")
            print("torch.compile: mode=reduce-overhead (Triton/inductor) ✓")
            return compiled
        except Exception as e:
            print(f"torch.compile reduce-overhead failed ({e}), trying aot_eager ...")

    try:
        compiled = torch.compile(model, backend="aot_eager")
        print("torch.compile: backend=aot_eager (no Triton — AOT graph capture) ✓")
        return compiled
    except Exception as e:
        print(f"torch.compile unavailable ({type(e).__name__}), running uncompiled.")
        return model

compiled_model = _try_compile(_base_model)


# ─────────────────────────────────────────────────────────────────────────────
# CACHE BUILDERS  (dict-based, num_workers=0)
# ─────────────────────────────────────────────────────────────────────────────
def build_augmented_cache(pairs, transform, n_versions=10, desc="Augmenting"):
    unique = list({p for pair in pairs for p in pair})
    cache  = {}
    for p in tqdm(unique, desc=f"  {desc}", leave=False):
        pil = image_cache[p]
        cache[p] = [transform(pil.copy()) for _ in range(n_versions)]
    return cache


def build_tensor_cache(pairs, transform, desc="Caching"):
    unique = list({p for pair in pairs for p in pair})
    cache  = {}
    for p in tqdm(unique, desc=f"  {desc}", leave=False):
        cache[p] = transform(image_cache[p].copy())
    return cache


print(f"\nBuilding train aug cache ({N_AUG} versions per image) ...")
train_aug_cache = build_augmented_cache(
    train_pairs, train_transform, n_versions=N_AUG, desc="Train")
_s       = train_aug_cache[next(iter(train_aug_cache))][0]
mb_train = (len(train_aug_cache) * N_AUG * _s.nelement() * _s.element_size()) / 1e6
print(f"  {len(train_aug_cache)} images × {N_AUG} = "
      f"{len(train_aug_cache)*N_AUG:,} tensors  ({mb_train:.0f} MB)")

print("Building val/test tensor caches ...")
val_tensor_cache  = build_tensor_cache(val_pairs,  test_transform, desc="Val")
test_tensor_cache = build_tensor_cache(test_pairs, test_transform, desc="Test")
mb = lambda c: sum(t.nelement()*t.element_size() for t in c.values()) / 1e6
print(f"  Val  : {len(val_tensor_cache)}  images  ({mb(val_tensor_cache):.0f} MB)")
print(f"  Test : {len(test_tensor_cache)} images  ({mb(test_tensor_cache):.0f} MB)")


# ─────────────────────────────────────────────────────────────────────────────
# DATASETS
# ─────────────────────────────────────────────────────────────────────────────
def _author_from_path(path):
    try:    return os.path.basename(path).split('_')[1]
    except: return "unknown"


class TrainPairDataset(Dataset):
    def __init__(self, pairs, labels, aug_cache, author2id, n_versions):
        self.pairs     = pairs
        self.labels    = labels
        self.aug_cache = aug_cache
        self.author2id = author2id
        self.n         = n_versions

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2 = self.pairs[idx]
        img1   = self.aug_cache[p1][random.randrange(self.n)]
        img2   = self.aug_cache[p2][random.randrange(self.n)]
        a1     = self.author2id.get(_author_from_path(p1), 0)
        a2     = self.author2id.get(_author_from_path(p2), 0)
        return (
            img1, img2,
            torch.tensor(self.labels[idx], dtype=torch.float32),
            torch.tensor(a1,               dtype=torch.long),
            torch.tensor(a2,               dtype=torch.long),
        )


class EvalPairDataset(Dataset):
    def __init__(self, pairs, labels, tensor_cache, author2id):
        self.pairs        = pairs
        self.labels       = labels
        self.tensor_cache = tensor_cache
        self.author2id    = author2id

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2 = self.pairs[idx]
        a1     = self.author2id.get(_author_from_path(p1), 0)
        a2     = self.author2id.get(_author_from_path(p2), 0)
        return (
            self.tensor_cache[p1], self.tensor_cache[p2],
            torch.tensor(self.labels[idx], dtype=torch.float32),
            torch.tensor(a1,               dtype=torch.long),
            torch.tensor(a2,               dtype=torch.long),
        )


# ─────────────────────────────────────────────────────────────────────────────
# LOSSES
# ─────────────────────────────────────────────────────────────────────────────
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, s=64.0, m=0.45):
        super().__init__()
        self.s = s; self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        W      = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings.float(), W.float())
        sine   = torch.sqrt((1.0 - cosine.pow(2)).clamp(1e-9, 1.0))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        oh     = torch.zeros_like(cosine)
        oh.scatter_(1, labels.view(-1, 1), 1.0)
        logits = (oh * phi + (1.0 - oh) * cosine) * self.s
        return F.cross_entropy(logits, labels, label_smoothing=0.1)


class FocalContrastiveLoss(nn.Module):
    """
    Euclidean contrastive loss with focal weighting.
    gamma=2: quadratically suppresses easy pairs, amplifies hard ones.
    """
    def __init__(self, margin=1.0, gamma=2.0):
        super().__init__()
        self.margin = margin
        self.gamma  = gamma

    def forward(self, emb1, emb2, labels):
        d     = F.pairwise_distance(emb1.float(), emb2.float(), p=2)
        pos   = d.pow(2)
        neg   = F.relu(self.margin - d).pow(2)
        pos_w = (d.detach() / self.margin).clamp(0, 1).pow(self.gamma)
        neg_w = (1 - d.detach() / self.margin).clamp(0, 1).pow(self.gamma)
        return (labels * pos_w * pos + (1 - labels) * neg_w * neg).mean()


class EmbeddingMixupLoss(nn.Module):
    """
    Manifold mixup in embedding space.
    Creates virtual hard examples at the genuine/forged decision boundary.
    lambda always > 0.5 so mixed embedding stays closer to genuine side.

    FIX #4: use torch.distributions.Beta instead of np.random.beta so that
    this loss is covered by torch.manual_seed and stays fully reproducible.
    """
    def __init__(self, margin=1.0, alpha=0.4):
        super().__init__()
        self.margin = margin
        self.alpha  = alpha

    def forward(self, emb1, emb2, labels):
        genuine_mask = labels.bool()
        forged_mask  = ~genuine_mask
        if genuine_mask.sum() < 1 or forged_mask.sum() < 1:
            return torch.tensor(0.0, device=emb1.device)

        e_genuine = torch.cat([emb1[genuine_mask], emb2[genuine_mask]], dim=0)
        e_forged  = torch.cat([emb1[forged_mask],  emb2[forged_mask]],  dim=0)
        n         = min(len(e_genuine), len(e_forged))
        e_genuine = e_genuine[:n]; e_forged = e_forged[:n]

        # FIX #4: torch Beta — same seed context as the rest of the model
        beta_dist = torch.distributions.Beta(
            torch.tensor(self.alpha), torch.tensor(self.alpha))
        lam = float(beta_dist.sample().item())
        lam = max(lam, 1 - lam)   # keep > 0.5 → stays on genuine side

        e_mix = F.normalize(lam * e_genuine + (1 - lam) * e_forged, p=2, dim=1)

        d_g = F.pairwise_distance(e_mix, e_genuine, p=2)
        d_f = F.pairwise_distance(e_mix, e_forged,  p=2)
        return (lam * d_g.pow(2) +
                (1 - lam) * F.relu(self.margin - d_f).pow(2)).mean()


class HardMiningLoss(nn.Module):
    """
    In-batch hard pair mining — zero extra forward passes.
    Hardest positive: genuine pair furthest apart.
    Hardest negative: forged pair closest together.

    FIX #3: divide by batch size (emb1.shape[0]) so this loss is on the
    same per-sample scale as FocalContrastiveLoss (~0.3–0.6).
    Without this, the loss was a sum of 2 raw squared distances (~50–200)
    versus contrastive mean (~0.5) — a ~100–400× imbalance at lambda_hard=0.25.
    """
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        d            = F.pairwise_distance(emb1.float(), emb2.float(), p=2)
        genuine_mask = labels.bool()
        forged_mask  = ~genuine_mask
        loss         = torch.tensor(0.0, device=emb1.device)

        if genuine_mask.sum() > 0:
            loss = loss + d[genuine_mask].max().pow(2)
        if forged_mask.sum() > 0:
            loss = loss + F.relu(self.margin - d[forged_mask].min()).pow(2)

        # FIX #3: normalise by batch size
        return loss / emb1.shape[0]


class SignatureVerificationLoss(nn.Module):
    """
    L = lambda_arc  * ArcFace          (raw, ~30–35 with s=64)
      + lambda_con  * FocalContrastive  (raw, ~0.3–0.6)
      + lambda_mix  * EmbeddingMixup
      + lambda_hard * HardMining        (now normalised by batch size)

    Scale reality with s=64:
      ArcFace ≈ 33,  Contrastive ≈ 0.5  →  ratio ≈ 66×
      lambda_arc=0.02 → 0.02×33 ≈ 0.66
      lambda_con=1.00 → 1.00×0.5 ≈ 0.50
      Both contribute ~equally.

    DO NOT divide losses by themselves — that always gives 1.0 (v8.1 bug).
    """
    def __init__(self, num_classes, emb_dim=256,
                 s=64.0, m=0.45, margin=1.0, gamma=2.0, mixup_alpha=0.4,
                 lambda_arc=0.02, lambda_con=1.0,
                 lambda_mix=0.1,  lambda_hard=0.1):
        super().__init__()
        self.arcface     = ArcFaceLoss(emb_dim, num_classes, s=s, m=m)
        self.contrastive = FocalContrastiveLoss(margin=margin, gamma=gamma)
        self.mixup       = EmbeddingMixupLoss(margin=margin, alpha=mixup_alpha)
        self.hardmine    = HardMiningLoss(margin=margin)
        self.la   = lambda_arc
        self.lc   = lambda_con
        self.lm   = lambda_mix
        self.lh   = lambda_hard

    def forward(self, emb1, emb2, pair_labels, auth1, auth2):
        arc  = (self.arcface(emb1, auth1) + self.arcface(emb2, auth2)) / 2.0
        con  = self.contrastive(emb1, emb2, pair_labels)
        mix  = self.mixup(emb1, emb2, pair_labels)
        hard = self.hardmine(emb1, emb2, pair_labels)

        total = self.la * arc + self.lc * con + self.lm * mix + self.lh * hard
        return total, arc.item(), con.item(), mix.item(), hard.item()


# ─────────────────────────────────────────────────────────────────────────────
# METRICS  (Euclidean distance)
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_distances(model, loader):
    model.eval()
    dists, labs = [], []
    for img1, img2, labels, _, _ in loader:
        with autocast('cuda', enabled=torch.cuda.is_available()):
            e1, e2 = model(img1.to(device, non_blocking=True),
                           img2.to(device, non_blocking=True))
        dists.append(F.pairwise_distance(e1.float(), e2.float()).cpu().numpy())
        labs.append(labels.numpy())
    return np.concatenate(dists), np.concatenate(labs)


def best_threshold(distances, labels):
    best_acc, best_t = 0.0, 0.5
    for t in np.linspace(distances.min(), distances.max(), 200):
        acc = accuracy_score(labels, (distances < t).astype(int))
        if acc > best_acc: best_acc, best_t = acc, t
    return best_t, best_acc


def evaluate(model, loader, split="Val"):
    dists, labels = compute_distances(model, loader)
    t, acc = best_threshold(dists, labels)
    try:    auc = roc_auc_score(labels, -dists)
    except: auc = 0.5
    preds   = (dists < t).astype(int)
    genuine = labels == 1; forged = labels == 0
    far     = float(np.mean(preds[forged]  == 1)) if forged.any()  else 0.0
    frr     = float(np.mean(preds[genuine] == 0)) if genuine.any() else 0.0
    gen_m   = float(dists[genuine].mean()) if genuine.any() else 0.0
    forg_m  = float(dists[forged].mean())  if forged.any()  else 0.0
    print(f"  [{split}] Acc={acc:.4f}  AUC={auc:.4f}  "
          f"FAR={far:.4f}  FRR={frr:.4f}  "
          f"Gap={forg_m-gen_m:.3f}  Gen={gen_m:.3f}  Forg={forg_m:.3f}  Thr={t:.4f}")
    return {"acc": acc, "auc": auc, "far": far, "frr": frr,
            "threshold": t, "gap": forg_m - gen_m}


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN EPOCH
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, criterion, optimizer, scheduler, scaler, loader, epoch):
    model.train()
    tot = tot_arc = tot_con = tot_mix = tot_hard = 0.0
    all_dists, all_labels = [], []
    use_amp = torch.cuda.is_available()
    pbar    = tqdm(loader, desc=f"Epoch {epoch:03d} [Train]", leave=False)

    for img1, img2, pair_labels, auth1, auth2 in pbar:
        img1        = img1.to(device, non_blocking=True)
        img2        = img2.to(device, non_blocking=True)
        pair_labels = pair_labels.to(device, non_blocking=True)
        auth1       = auth1.to(device, non_blocking=True)
        auth2       = auth2.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=use_amp):
            e1, e2 = model(img1, img2)
            loss, arc, con, mix, hard = criterion(
                e1, e2, pair_labels, auth1, auth2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()

        with torch.no_grad():
            d = F.pairwise_distance(e1.float(), e2.float())
            all_dists.append(d.cpu().numpy())
            all_labels.append(pair_labels.cpu().numpy())

        tot += loss.item(); tot_arc += arc; tot_con += con
        tot_mix += mix; tot_hard += hard
        pbar.set_postfix(loss=f"{loss.item():.3f}",
                 arc=f"{arc:.1f}", con=f"{con:.3f}",
                 mix=f"{mix:.3f}", hard=f"{hard:.3f}")

    n = len(loader)
    all_dists  = np.concatenate(all_dists)
    all_labels = np.concatenate(all_labels)
    _, tr_acc  = best_threshold(all_dists, all_labels)

    # Weighted ratio = (lambda_arc * arc) / (lambda_con * con)
    # This is the TRUE gradient balance — raw ratio is always ~500x and meaningless.
    w_arc  = criterion.la * (tot_arc / n)
    w_con  = criterion.lc * (tot_con / n)
    w_mix  = criterion.lm * (tot_mix / n)
    w_hard = criterion.lh * (tot_hard / n)
    w_ratio = (w_arc / w_con) if w_con > 1e-9 else float('inf')
    # Print the end-of-epoch LR for both param groups (model + arcface head)
    lrs = [pg["lr"] for pg in optimizer.param_groups]
    lr_str = "/".join(f"{lr:.2e}" for lr in lrs)
    print(f"  [Train] Loss={tot/n:.4f}  "
          f"Arc={tot_arc/n:.2f}(w={w_arc:.3f})  Con={tot_con/n:.4f}(w={w_con:.3f})  "
          f"Mix={tot_mix/n:.4f}(w={w_mix:.3f})  Hard={tot_hard/n:.4f}(w={w_hard:.3f})  "
          f"wRatio={w_ratio:.1f}x  LR={lr_str}  Acc={tr_acc:.4f}")
    return tot / n, tr_acc


# ─────────────────────────────────────────────────────────────────────────────
# VAL EPOCH
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def val_epoch(model, criterion, loader):
    model.eval()
    tot    = 0.0
    use_amp = torch.cuda.is_available()
    for img1, img2, pair_labels, auth1, auth2 in loader:
        img1        = img1.to(device, non_blocking=True)
        img2        = img2.to(device, non_blocking=True)
        pair_labels = pair_labels.to(device, non_blocking=True)
        auth1       = auth1.to(device, non_blocking=True)
        auth2       = auth2.to(device, non_blocking=True)
        with autocast('cuda', enabled=use_amp):
            e1, e2 = model(img1, img2)
            loss, _, _, _, _ = criterion(e1, e2, pair_labels, auth1, auth2)
        tot += loss.item()
    print(f"  [Val  ] Loss={tot/len(loader):.4f}")
    return tot / len(loader)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def train(
    model,
    num_epochs     = 60,
    lr             = 2e-4,
    weight_decay   = 5e-4,
    arcface_scale  = 32.0,   # was 64 — s=64 causes ArcFace to converge in ~10 epochs then stall
    arcface_margin = 0.45,
    margin         = 1.0,
    gamma          = 2.5,
    mixup_alpha    = 0.35,
    lambda_arc     = 0.015,
    lambda_con     = 2.5,
    lambda_mix     = 0.25,
    lambda_hard    = 0.25,
    save_dir       = "./checkpoints",
    patience       = 12,
):
    os.makedirs(save_dir, exist_ok=True)

    # FIX: only train-split authors — val/test authors were never seen by ArcFace,
    # including them added dead weight rows that never received gradients.
    author2id   = {a: i for i, a in enumerate(sorted(train_authors))}
    num_classes = len(author2id)
    print(f"\nAuthors (ArcFace classes) : {num_classes} (train only — no val/test leakage)")

    train_ds = TrainPairDataset(
        train_pairs, train_labels, train_aug_cache, author2id, N_AUG)
    val_ds   = EvalPairDataset(
        val_pairs,   val_labels,   val_tensor_cache,  author2id)
    test_ds  = EvalPairDataset(
        test_pairs,  test_labels,  test_tensor_cache, author2id)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,
                              num_workers=0, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                              num_workers=0, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False,
                              num_workers=0, pin_memory=pin)

    print(f"Loaders  train:{len(train_loader)}  "
          f"val:{len(val_loader)}  test:{len(test_loader)} batches")

    criterion = SignatureVerificationLoss(
        num_classes, emb_dim=256,
        s=arcface_scale, m=arcface_margin,
        margin=margin, gamma=gamma, mixup_alpha=mixup_alpha,
        lambda_arc=lambda_arc, lambda_con=lambda_con,
        lambda_mix=lambda_mix, lambda_hard=lambda_hard,
    ).to(device)

    optimizer = AdamW([
        {"params": model.parameters(),
         "lr": lr, "weight_decay": weight_decay},
        {"params": criterion.arcface.parameters(),
         "lr": lr * 0.1, "weight_decay": weight_decay},
    ])

    # CosineAnnealingWarmRestarts: periodic LR restarts every T_0 epochs.
    # When val loss starts rising, the next restart bumps LR back up and
    # forces re-exploration instead of the model memorising training pairs.
    # T_0=10: restart every 10 epochs. T_mult=2: each cycle doubles (10,20,40...).
    # eta_min: floor LR so gradients never fully die.
    # OneCycleLR was monotonically decaying — no recovery once overfitting started.
    steps_per_epoch = len(train_loader)
    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=15 * steps_per_epoch,   # restart every 15 epochs — 10 was too early, disturbed training
        T_mult=2,                    # double cycle length after each restart
        eta_min=lr * 0.01,           # floor = 2e-6 (stays non-zero)
    )

    scaler   = GradScaler('cuda', enabled=torch.cuda.is_available())
    best_auc = 0.0; best_epoch = 0; no_improve = 0
    history  = {"train_loss": [], "train_acc": [],
                "val_loss":   [], "val_auc":   [], "val_acc": []}

    print("\n" + "=" * 70)
    print("TRAINING  —  ArcFace + FocalContrastive(L2) + Mixup + HardMine")
    print(f"  λ_arc={lambda_arc}  λ_con={lambda_con}  "
          f"λ_mix={lambda_mix}  λ_hard={lambda_hard}")
    print(f"  Expected total loss ≈ "
          f"{lambda_arc*33:.2f} + {lambda_con*0.5:.2f} = "
          f"{lambda_arc*33 + lambda_con*0.5:.2f}  (NOT 1.0)")
    print(f"  Cache refresh every {CACHE_REFRESH} epochs  |  "
          f"{N_AUG} aug versions/img")
    print(f"  Epochs:{num_epochs}  LR:{lr}  Patience:{patience}")
    print("=" * 70)

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        print(f"\nEpoch {epoch}/{num_epochs}")

        if epoch > 1 and (epoch - 1) % CACHE_REFRESH == 0:
            print(f"  ↻ Refreshing aug cache ...")
            new_cache = build_augmented_cache(
                train_pairs, train_transform, n_versions=N_AUG, desc="Refresh")
            train_ds.aug_cache = new_cache

        tl, tr_acc = train_epoch(model, criterion, optimizer, scheduler,
                                  scaler, train_loader, epoch)
        vl = val_epoch(model, criterion, val_loader)
        vm = evaluate(model, val_loader, "Val")
        print(f"  Time: {time.time()-t0:.0f}s")

        history["train_loss"].append(tl)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl)
        history["val_auc"].append(vm["auc"])
        history["val_acc"].append(vm["acc"])

        if vm["auc"] > best_auc:
            best_auc, best_epoch, no_improve = vm["auc"], epoch, 0
            torch.save({
                "epoch":       epoch,
                "model_state": _base_model.state_dict(),   # save base, not compiled
                "arc_state":   criterion.arcface.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "scheduler":   scheduler.state_dict(),
                "val_auc":     best_auc,
                "val_acc":     vm["acc"],
                "threshold":   vm["threshold"],
                "author2id":   author2id,
            }, os.path.join(save_dir, "best_model.pt"))
            print(f"  ✓ Best  AUC={best_auc:.4f}  Acc={vm['acc']:.4f}")
        else:
            no_improve += 1
            print(f"  No improve {no_improve}/{patience}")
            if no_improve >= patience:
                print(f"  Early stop at epoch {epoch}"); break

    # ── Test ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 70 + "\nFINAL TEST EVALUATION\n" + "=" * 70)
    ckpt = torch.load(os.path.join(save_dir, "best_model.pt"),
                      map_location=device, weights_only=False)
    _base_model.load_state_dict(ckpt["model_state"])
    # FIX: evaluate on _base_model directly — compiled_model wraps _base_model
    # but torch.compile can cache stale kernel state after in-place weight update.
    # _base_model always reflects the freshly loaded weights.
    _base_model.eval()
    tm = evaluate(_base_model, test_loader, "Test")

    print(f"\nBest epoch={best_epoch}  Val AUC={best_auc:.4f}")
    print(f"Test  AUC={tm['auc']:.4f}  Acc={tm['acc']:.4f}  "
          f"FAR={tm['far']:.4f}  FRR={tm['frr']:.4f}  Gap={tm['gap']:.3f}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    dists, labels_np = compute_distances(_base_model, test_loader)
    gen_d  = dists[labels_np == 1]
    forg_d = dists[labels_np == 0]

    try:
        import seaborn as sns
        plt.figure(figsize=(9, 5))
        sns.histplot(gen_d,  color='royalblue', label='Genuine',
                     kde=True, stat='density', alpha=0.6)
        sns.histplot(forg_d, color='tomato',    label='Forged',
                     kde=True, stat='density', alpha=0.6)
    except ImportError:
        plt.figure(figsize=(9, 5))
        plt.hist(gen_d,  bins=50, color='royalblue',
                 label='Genuine', density=True, alpha=0.6)
        plt.hist(forg_d, bins=50, color='tomato',
                 label='Forged',  density=True, alpha=0.6)

    plt.axvline(tm["threshold"], color='green', linestyle='--',
                label=f"Thr ({tm['threshold']:.3f})")
    plt.xlabel('Euclidean Distance'); plt.ylabel('Density')
    plt.title('Test Distance Distribution')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "distance_distribution.png"), dpi=150)
    plt.close()

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].plot(history["train_acc"], label="Train", color='steelblue')
    axes[1].plot(history["val_acc"],   label="Val",   color='coral')
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
    axes[2].plot(history["val_auc"])
    axes[2].set_title("Val AUC"); axes[2].set_xlabel("Epoch")
    axes[3].plot(history["val_acc"])
    axes[3].set_title("Val Accuracy"); axes[3].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150)
    plt.close()

    print(f"Plots → {save_dir}/")
    return compiled_model, history, tm


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    trained_model, history, test_metrics = train(
        model          = compiled_model,
        num_epochs     = 60,
        lr             = 2e-4,
        weight_decay   = 5e-4,
        arcface_scale  = 64.0,
        arcface_margin = 0.45,
        margin         = 1.0,
        gamma          = 2.5,
        mixup_alpha    = 0.35,
        lambda_arc     = 0.015,
        lambda_con     = 2.5,    # target wRatio ~2x  (0.015×32 / 2.5×0.06 ≈ 3.2x)
        lambda_mix     = 0.25,
        lambda_hard    = 0.25,
        save_dir       = "./checkpoints",
        patience       = 12,
    )

Genuine authors : 268
Forgery authors : 276
  Author 001: 24 genuine signatures
  Author 002: 24 genuine signatures
  Author 003: 24 genuine signatures
  Author 004: 24 genuine signatures
  Author 006: 24 genuine signatures
  Author 009: 24 genuine signatures
  Author 012: 24 genuine signatures
  Author 014: 24 genuine signatures
  Author 015: 24 genuine signatures
  Author 016: 23 genuine signatures
  Author 021: 24 genuine signatures
  Author 022: 24 genuine signatures
  Author 023: 24 genuine signatures
  Author 024: 24 genuine signatures
  Author 025: 24 genuine signatures
  Author 026: 24 genuine signatures
  Author 027: 24 genuine signatures
  Author 028: 24 genuine signatures
  Author 029: 24 genuine signatures
  Author 036: 24 genuine signatures
  Author 037: 24 genuine signatures
  Author 038: 24 genuine signatures
  Author 039: 24 genuine signatures
  Author 040: 24 genuine signatures
  Author 041: 24 genuine signatures
  Author 042: 24 genuine signatures
  Author 043: 24 gen

  5671 images × 15 = 85,065 tensors  (3136 MB)
Building val/test tensor caches ...


  Val  : 791  images  (29 MB)
  Test : 860 images  (32 MB)

Authors (ArcFace classes) : 214 (train only — no val/test leakage)
Loaders  train:236  val:19  test:19 batches

TRAINING  —  ArcFace + FocalContrastive(L2) + Mixup + HardMine
  λ_arc=0.015  λ_con=2.5  λ_mix=0.25  λ_hard=0.25
  Expected total loss ≈ 0.49 + 1.25 = 1.75  (NOT 1.0)
  Cache refresh every 5 epochs  |  15 aug versions/img
  Epochs:60  LR:0.0002  Patience:12

Epoch 1/60


  [Train] Loss=0.6235  Arc=31.85(w=0.478)  Con=0.0515(w=0.129)  Mix=0.0649(w=0.016)  Hard=0.0037(w=0.001)  wRatio=3.7x  LR=1.98e-04/1.98e-05  Acc=0.6237
  [Val  ] Loss=0.9054
  [Val] Acc=0.7770  AUC=0.8460  FAR=0.2844  FRR=0.1616  Gap=0.251  Gen=0.098  Forg=0.349  Thr=0.1600
  Time: 164s
  ✓ Best  AUC=0.8460  Acc=0.7770

Epoch 2/60


  [Train] Loss=0.5622  Arc=30.09(w=0.451)  Con=0.0373(w=0.093)  Mix=0.0671(w=0.017)  Hard=0.0037(w=0.001)  wRatio=4.8x  LR=1.91e-04/1.92e-05  Acc=0.6949
  [Val  ] Loss=0.7609
  [Val] Acc=0.8320  AUC=0.9114  FAR=0.2050  FRR=0.1310  Gap=0.286  Gen=0.118  Forg=0.404  Thr=0.1918
  Time: 92s
  ✓ Best  AUC=0.9114  Acc=0.8320

Epoch 3/60


...

  [Train] Loss=0.3907  Arc=19.74(w=0.296)  Con=0.0323(w=0.081)  Mix=0.0515(w=0.013)  Hard=0.0037(w=0.001)  wRatio=3.7x  LR=1.98e-04/1.98e-05  Acc=0.8169
  [Val  ] Loss=0.7061
  [Val] Acc=0.8647  AUC=0.9386  FAR=0.1668  FRR=0.1038  Gap=0.382  Gen=0.340  Forg=0.723  Thr=0.4861
  Time: 90s
  No improve 8/12

Epoch 18/60


  [Train] Loss=0.3560  Arc=17.13(w=0.257)  Con=0.0348(w=0.087)  Mix=0.0443(w=0.011)  Hard=0.0039(w=0.001)  wRatio=2.9x  LR=1.95e-04/1.96e-05  Acc=0.8148
  [Val  ] Loss=0.7353
  [Val] Acc=0.8605  AUC=0.9362  FAR=0.1858  FRR=0.0932  Gap=0.382  Gen=0.346  Forg=0.728  Thr=0.5047
  Time: 90s
  No improve 9/12

Epoch 19/60


  [Train] Loss=0.3305  Arc=15.02(w=0.225)  Con=0.0367(w=0.092)  Mix=0.0494(w=0.012)  Hard=0.0039(w=0.001)  wRatio=2.5x  LR=1.91e-04/1.92e-05  Acc=0.8106
  [Val  ] Loss=0.7637
  [Val] Acc=0.8488  AUC=0.9290  FAR=0.1886  FRR=0.1139  Gap=0.382  Gen=0.372  Forg=0.754  Thr=0.5321
  Time: 90s
  No improve 10/12

Epoch 20/60


  [Train] Loss=0.3054  Arc=13.21(w=0.198)  Con=0.0386(w=0.096)  Mix=0.0392(w=0.010)  Hard=0.0040(w=0.001)  wRatio=2.1x  LR=1.87e-04/1.88e-05  Acc=0.8033
  [Val  ] Loss=0.7562
  [Val] Acc=0.8335  AUC=0.9142  FAR=0.2235  FRR=0.1094  Gap=0.377  Gen=0.412  Forg=0.789  Thr=0.5948
  Time: 90s
  No improve 11/12

Epoch 21/60
  ↻ Refreshing aug cache ...


  [Train] Loss=0.2901  Arc=11.99(w=0.180)  Con=0.0390(w=0.097)  Mix=0.0470(w=0.012)  Hard=0.0042(w=0.001)  wRatio=1.8x  LR=1.81e-04/1.83e-05  Acc=0.7992
  [Val  ] Loss=0.8009
  [Val] Acc=0.8557  AUC=0.9305  FAR=0.1797  FRR=0.1090  Gap=0.396  Gen=0.405  Forg=0.801  Thr=0.5795
  Time: 253s
  No improve 12/12
  Early stop at epoch 21

FINAL TEST EVALUATION
  [Test] Acc=0.8680  AUC=0.9347  FAR=0.1505  FRR=0.1134  Gap=0.399  Gen=0.262  Forg=0.661  Thr=0.3823

Best epoch=9  Val AUC=0.9547
Test  AUC=0.9347  Acc=0.8680  FAR=0.1505  FRR=0.1134  Gap=0.399
Plots → ./checkpoints/
